-----
# Get indications from combined jsons

In [2]:
import requests
import json
import pandas as pd
from pathlib import Path
import os

In [3]:

# Load all JSON files in the directory and merge their content
def load_and_combine_json(directory):
    combined = []
    for filename in os.listdir(directory):
        if filename.endswith('.json'):
            with open(os.path.join(directory, filename), 'r', encoding='utf-8') as file:
                content = json.load(file)

                # If each file is a list of records (e.g. [ {...}, {...} ])
                if isinstance(content, list):
                    combined.extend(content)
                # If each file is a single JSON object (e.g. { ... })
                elif isinstance(content, dict):
                    combined.append(content)

    return combined


# # Define your directory
# directory = Path('C:/Projects/DrugFork/data/FDA/labels')
# combined_data = load_and_combine_json(directory)

# # Save to a new consolidated JSON file
# output_path = Path('C:/Projects/DrugFork/data/FDA/combined_labels.json')
# with open(output_path, 'w', encoding='utf-8') as f:
#     json.dump(combined_data, f, indent=4)

# print(f"Combined JSON saved to: {output_path}")


# Get unique numbers to search indications for

In [24]:
with open("data/FDA/formatted_output_openFDA.json", "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(len(df))

unique_marketing_authorisation_numbers = df["Marketing authorisation Number"].unique()
len(unique_marketing_authorisation_numbers)
unique_marketing_authorisation_numbers[:50]

28288


array(['ANDA076177', 'ANDA076178', 'ANDA076183', 'ANDA076213',
       'ANDA076215', 'ANDA076221', 'ANDA076254', 'NDA017391',
       'ANDA076258', 'ANDA076273', 'ANDA076298', 'ANDA076300',
       'ANDA076302', 'ANDA076304', 'NDA017406', 'ANDA076312',
       'ANDA076324', 'ANDA076332', 'ANDA076338', 'ANDA076342',
       'ANDA076363', 'ANDA076372', 'ANDA076388', 'ANDA076405',
       'NDA017423', 'ANDA076416', 'NDA017424', 'ANDA076442', 'ANDA076461',
       'ANDA076498', 'ANDA076505', 'ANDA076510', 'ANDA076534',
       'ANDA076554', 'ANDA076555', 'ANDA076561', 'ANDA076591',
       'ANDA076593', 'ANDA076607', 'ANDA076608', 'ANDA076616',
       'ANDA076624', 'NDA017451', 'ANDA076633', 'ANDA076635',
       'ANDA076652', 'ANDA076667', 'ANDA076668', 'ANDA076671',
       'ANDA076673'], dtype=object)

In [5]:
# Load the merged local FDA JSON file once
with open("C:/Projects/DrugFork/data/FDA/combined_labels.json", "r", encoding="utf-8") as f:
    fda_data = json.load(f)

In [19]:
flattened_fda_data = [entry
    for batch in fda_data
    for entry in batch.get("results", [])
]
# flattened_fda_data[:2]  # Display first two entries for inspection

for entry in flattened_fda_data[:5]:
    print(entry.get("openfda", {}).get("application_number"))
    print(entry.get("openfda", {}).get("brand_name"))


None
['UNDA 312']
['NDA215179']
['PEMRYDI RTU']
['ANDA217340']
['Guaifenesin and Dextromethorphan HBR']
['ANDA210651']
['Dextroamphetamine Saccharate, Amphetamine Aspartate Monohydrate, Dextroamphetamine Sulfate, and Amphetamine Sulfate Extended-Release']
['M020']
['CYZONE CY PLAY CREAMY LIP BALM HIDRATANTE EN BARRA PARA LABIOS 24H FPS 12 FUCHSIA CREAMY']


In [ ]:
# Define search function using local data
def search_drug_by_number(marketing_authorisation_number, combined_fda):
    for entry in combined_fda:
        app_numbers = entry.get("openfda", {}).get("application_number", [])

        if marketing_authorisation_number in app_numbers:
            return entry
    print(f"No results found for {marketing_authorisation_number}")
    return None



all_results = []

for n in unique_marketing_authorisation_numbers:
    result = search_drug_by_number(n, flattened_fda_data)
    if result:
        all_results.append({
            "Marketing Authorisation Number": n,
            "Indications and Usage": result.get("indications_and_usage", ["No indication found."])[0]
        })

        # Save intermediate results
        with open("data/FDA/indications_and_usage.json", "w", encoding="utf-8") as f:
            json.dump(all_results, f, indent=4)

# Export final results
df = pd.DataFrame(all_results)
df.to_csv("data/FDA/indications_and_usage.csv", index=False)

No results found for ANDA076213
No results found for ANDA076215
No results found for NDA017391
No results found for ANDA076298
No results found for ANDA076300
No results found for ANDA076302
No results found for ANDA076304
No results found for NDA017406
No results found for ANDA076312
No results found for ANDA076324
No results found for ANDA076332
No results found for ANDA076338
No results found for ANDA076372
No results found for ANDA076388
No results found for ANDA076405
No results found for NDA017423
No results found for NDA017424
No results found for ANDA076461
No results found for ANDA076498
No results found for ANDA076505
No results found for ANDA076555
No results found for ANDA076591
No results found for ANDA076607
No results found for ANDA076608
No results found for NDA017451
No results found for ANDA076633
No results found for ANDA076635
No results found for ANDA076652
No results found for ANDA076671
No results found for ANDA076673
No results found for ANDA076675
No results fo

----
# Combine all FDA data

In [23]:
# open json file to inspect
with open("data/FDA/indications_and_usage.json", "r", encoding="utf-8") as f:
    indications_data = json.load(f)

with open("data/FDA/formatted_output_openFDA_manual_orphan_drug.json", "r", encoding="utf-8") as f:
    orphan_drug_data = json.load(f)

df_orphan_matched = pd.read_csv("data/FDA/orphan_drugs_matched.csv", dtype=str, encoding="utf-8")

indications_df = pd.DataFrame(indications_data)
orphan_drug_df = pd.DataFrame(orphan_drug_data)

In [24]:
df_orphan_matched.head()

,Generic Name,Trade Name,Date Designated,Orphan Designation,Orphan Designation Status,Date Designation Withdrawn or Revoked,FDA Orphan Approval Status,Approved Labeled Indication,Marketing Approval Date,Exclusivity End Date,"Exclusivity Protected Indication * (Shown for approvals from Jan. 1, 2013, to the present)",Sponsor Company,Sponsor Address 1,Sponsor Address 2,Sponsor City,Sponsor State,Sponsor Zip,Sponsor Country,CF Grid Key
0,bosentan,Tracleer,2000-06-10 00:00:00,Treatment of pulmonary arterial hypertension,Designated/Approved,NaN,NaN,Treatment of pulmonary arterial hypertension (...,2017-05-09 00:00:00,2024-05-09 00:00:00,Treatment of pulmonary arterial hypertension (...,Actelion Pharmaceuticals Ltd,1840 Gateway Drive,Suite 300,Cherry Hill,New Jersey,' 08002 ',United States,134200.0
1,bosentan,Tracleer,2000-06-10 00:00:00,Treatment of pulmonary arterial hypertension,Designated/Approved,NaN,NaN,Treatment of pulmonary arterial hypertension.,11/20/2001,11/20/2008,NaN,Actelion Pharmaceuticals Ltd,1840 Gateway Drive,Suite 300,Cherry Hill,New Jersey,' 08002 ',United States,134200.0
2,5-aminolevulinic acid,Gleolan,01/15/2013,Visualization of malignant tissue during surge...,Designated/Approved,NaN,NaN,Optical imaging agent indicated in patients wi...,2017-06-06 00:00:00,2024-06-06 00:00:00,Optical imaging agent indicated in patients wi...,NX Development Corporation,1827 South Bayshore Lane,NaN,Miami,Florida,' 33133 ',United States,387312.0
3,abatacept,Orencia,12/26/2017,Prevention of graft versus host disease,Designated/Approved,NaN,NaN,prophylaxis of acute graft versus host disease...,12/15/2021,12/15/2028,prophylaxis of acute graft versus host disease...,Bristol-Myers Squibb Co.,P. O. Box 5326,NaN,Princeton,New Jersey,' 08543 ',United States,614117.0
4,acalabrutinib,Calquence,05/13/2015,Treatment of chronic lymphocytic leukemia (CLL).,Designated/Approved,NaN,NaN,Treatment of adult patients with chronic lymph...,2022-04-08 00:00:00,NaN,NaN,"Acerta Pharma, LLC (a member of the AstraZenec...",121 Oyster Point Boulevard,NaN,South San Francisco,California,' 94080 ',United States,477415.0


In [25]:
indications_df = indications_df.rename(columns={"Marketing Authorisation Number": "MA_Number"})
indications_df.columns
indications_df["MA_Number"].nunique() == len(indications_df)
indications_df.head()

,MA_Number,Indications and Usage
0,ANDA076177,INDICATIONS AND USAGE 1. Indications Progestin...
1,ANDA076178,INDICATIONS AND USAGE Nizatidine capsules are ...
2,ANDA076183,1 INDICATIONS AND USAGE Ondansetron tablets ar...
3,ANDA076221,1 INDICATIONS AND USAGE Jinteli is a combinati...
4,ANDA076254,1 INDICATIONS AND USAGE Brimonidine tartrate o...


In [26]:
orphan_drug_df = orphan_drug_df.rename(columns={"Marketing authorisation Number": "MA_Number"})
orphan_drug_df.columns
orphan_drug_df["MA_Number"].nunique() == len(orphan_drug_df)
orphan_drug_df.head()

,Origin,MA_Number,Drug name,Non proprietary name,Marketing authorisation holder/ applicant,Pharmaceutical form,Administration route,Decision,Decision date,Current status,Non-clinical abridge,Referral,Orphan_drug_status
0,FDA,ANDA076177,CAMILA,NORETHINDRONE,DR REDDYS LABS SA,TABLET,ORAL-28,Approved,21.10.2002,authorised,yes,No,NaN
1,FDA,ANDA076178,NIZATIDINE,NIZATIDINE,EPIC PHARMA LLC,CAPSULE,ORAL,Approved,05.07.2002,authorised,yes,No,NaN
2,FDA,ANDA076183,ONDANSETRON HYDROCHLORIDE,ONDANSETRON HYDROCHLORIDE,DR REDDYS LABS LTD,TABLET,ORAL,Approved,26.12.2006,authorised,yes,No,NaN
3,FDA,ANDA076213,FLUCONAZOLE,FLUCONAZOLE,ROXANE,TABLET,ORAL,Approved,29.07.2004,Discontinued,yes,No,NaN
4,FDA,ANDA076215,BETAMETHASONE DIPROPIONATE,BETAMETHASONE DIPROPIONATE,FOUGERA PHARMS,"CREAM, AUGMENTED",TOPICAL,Approved,09.12.2003,authorised,yes,No,NaN


In [27]:
# merge dfs by "MA_Number"
merged_df = pd.merge(indications_df, orphan_drug_df, on="MA_Number", how="outer")
merged_df.head()


,MA_Number,Indications and Usage,Origin,Drug name,Non proprietary name,Marketing authorisation holder/ applicant,Pharmaceutical form,Administration route,Decision,Decision date,Current status,Non-clinical abridge,Referral,Orphan_drug_status
0,ANDA018659,NaN,FDA,ALLOPURINOL,ALLOPURINOL,MYLAN,TABLET,ORAL,Approved,24.10.1986,authorised,yes,No,NaN
1,ANDA020167,NaN,FDA,ESTRADIOL,ESTRADIOL,MYLAN PHARMS INC,SYSTEM,TRANSDERMAL,Approved,19.12.2014,authorised,yes,TBD,NaN
2,ANDA020345,NaN,FDA,AMINOSYN-HF 8%,AMINO ACIDS,ICU MEDICAL INC,INJECTABLE,INJECTION,Approved,04.04.1996,Discontinued,yes,No,NaN
3,ANDA020360,NaN,FDA,HEPATASOL 8%,AMINO ACIDS,BAXTER HLTHCARE,INJECTABLE,INJECTION,Approved,04.04.1996,Discontinued,yes,No,NaN
4,ANDA020374,NaN,FDA,INPERSOL-LC/LM W/ DEXTROSE 1.5% IN PLASTIC CON...,CALCIUM CHLORIDE,FRESENIUS,SOLUTION,INTRAPERITONEAL,Approved,13.06.1994,Discontinued,yes,No,NaN


In [37]:
# merged_df["Orphan_drug_status_new"] 
for row in merged_df.iterrows():
    index, data = row
    if data["Orphan_drug_status"] == "yes" or data["Orphan_drug_status"] == "Yes":
        merged_df.at[index, "Orphan_drug_status_new"] = "Yes"
    else:
        if data["Drug name"] in df_orphan_matched["Trade Name"].values:
            merged_df.at[index, "Orphan_drug_status_new"] = "Yes"
        else:
            merged_df.at[index, "Orphan_drug_status_new"] = "No"


In [38]:
merged_df.head(-50)

,MA_Number,Indications and Usage,Origin,Drug name,Non proprietary name,Marketing authorisation holder/ applicant,Pharmaceutical form,Administration route,Decision,Decision date,Current status,Non-clinical abridge,Referral,Orphan_drug_status,Orphan_drug_status_new
0,ANDA018659,NaN,FDA,ALLOPURINOL,ALLOPURINOL,MYLAN,TABLET,ORAL,Approved,24.10.1986,authorised,yes,No,NaN,No
1,ANDA020167,NaN,FDA,ESTRADIOL,ESTRADIOL,MYLAN PHARMS INC,SYSTEM,TRANSDERMAL,Approved,19.12.2014,authorised,yes,TBD,NaN,No
2,ANDA020345,NaN,FDA,AMINOSYN-HF 8%,AMINO ACIDS,ICU MEDICAL INC,INJECTABLE,INJECTION,Approved,04.04.1996,Discontinued,yes,No,NaN,No
3,ANDA020360,NaN,FDA,HEPATASOL 8%,AMINO ACIDS,BAXTER HLTHCARE,INJECTABLE,INJECTION,Approved,04.04.1996,Discontinued,yes,No,NaN,No
4,ANDA020374,NaN,FDA,INPERSOL-LC/LM W/ DEXTROSE 1.5% IN PLASTIC CON...,CALCIUM CHLORIDE,FRESENIUS,SOLUTION,INTRAPERITONEAL,Approved,13.06.1994,Discontinued,yes,No,NaN,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28233,NDA218490,NaN,FDA,OPSYNVI,MACITENTAN,ACTELION,TABLET,ORAL,Approved,22.03.2024,authorised,no,Yes,NaN,No
28234,NDA218527,NaN,FDA,CHEWTADZY,TADALAFIL,B BETTER,"TABLET, CHEWABLE",ORAL,Approved,28.06.2024,Discontinued,no,Yes,NaN,No
28235,NDA218549,NaN,FDA,ZUNVEYL,BENZGALANTAMINE GLUCONATE,ALPHA COGNITION,"TABLET, DELAYED RELEASE",ORAL,Approved,26.07.2024,authorised,no,Yes,NaN,No
28236,NDA218550,NaN,FDA,ROZLYTREK,ENTRECTINIB,GENENTECH INC,PELLETS,ORAL,Approved,20.10.2023,authorised,no,Yes,NaN,Yes


In [39]:
merged_df["Orphan_drug_status_new"].value_counts()


Orphan_drug_status_new
No     28119
Yes      169
Name: count, dtype: int64

In [40]:
merged_df.to_csv("data/FDA/Merged_with_indications_and_orphan.csv", index=False, encoding="utf-8")   

------
# Scraping experiments

In [7]:
def search_drug_by_number(marketing_authorisation_number):
    url = "https://api.fda.gov/drug/label.json"
    params = {
        "search": f'openfda.application_number:"{marketing_authorisation_number}"'
    }

    response = requests.get(url, params=params)
    try:
        data = response.json()
        if not data or "results" not in data:
            print(f"No results found for {marketing_authorisation_number}")
            return None
        print(data)
        if data["results"]:

            indications = data["results"][0].get("indications_and_usage", ["No indication found."])
            # print(f"\nApplication Number: {marketing_authorisation_number}\nIndication:\n{indications[0]}")
    except requests.exceptions.HTTPError as err:
        print(f"HTTP error: {err}")
    except Exception as e:
        print(f"Error: {e}")

    with open("result.txt", "w", encoding="utf-8") as f:
        f.write(str(result))

    return result


# all_results =  []

# for n in unique_marketing_authorisation_numbers[:3]:
#     result = search_drug_by_number(n)
#     if result:
#         all_results.append({
#             "Marketing authorisation Number": n,
#             "Indications and Usage": result.get("indications_and_usage", ["No indication found."])[0]
#         })
#         # save intermediate results to json
#         with open("data/FDA/indications_and_usage.json", "w", encoding="utf-8") as f:
#             json.dump(all_results, f, indent=4)


# df = pd.DataFrame(all_results)
# df.to_csv("data/FDA/indications_and_usage.csv", index=False)

